In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamp as UTC datetimes
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Compute overall timestamp ranges (min/max) for each file as compact DataFrames
metric_range = pd.DataFrame({
    "file": ["metric.csv"],
    "min_ts": [df_metric['timestamp'].min()],
    "max_ts": [df_metric['timestamp'].max()]
})

trace_range = pd.DataFrame({
    "file": ["trace.csv"],
    "min_ts": [df_trace['timestamp'].min()],
    "max_ts": [df_trace['timestamp'].max()]
})

log_range = pd.DataFrame({
    "file": ["log.csv"],
    "min_ts": [df_log['timestamp'].min()],
    "max_ts": [df_log['timestamp'].max()]
})

error_range = pd.DataFrame({
    "file": ["error_logs.csv"],
    "min_ts": [df_error['timestamp'].min()],
    "max_ts": [df_error['timestamp'].max()]
})

# 1) metric.csv aggregation by (cmdb_id, kpi_name): count and global P95 of 'value'
metric_agg = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count', p95=lambda s: s.quantile(0.95))
    .sort_values('count', ascending=False)
)
metric_agg = metric_agg[['cmdb_id', 'kpi_name', 'count', 'p95']].head(50)

# 2) trace.csv aggregation by (cmdb_id, trace_name): count and global P95 of 'value'
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count', p95=lambda s: s.quantile(0.95))
    .sort_values('count', ascending=False)
)
trace_agg = trace_agg[['cmdb_id', 'trace_name', 'count', 'p95']].head(50)

# 3) log.csv aggregation by (cmdb_id, log_name): count and global P95 of 'value'
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count', p95=lambda s: s.quantile(0.95))
    .sort_values('count', ascending=False)
)
log_agg = log_agg[['cmdb_id', 'log_name', 'count', 'p95']].head(50)

# 4) error_logs.csv: sample up to first 10 rows (all columns)
error_sample = df_error.head(10)

# Compact outputs: timestamp ranges and aggregated tables/samples
metric_range, metric_agg, trace_range, trace_agg, log_range, log_agg, error_range, error_sample

```
Out[1]:
```
```python
# Compose a plain-English summary based on previous aggregations available in the kernel
# Reuse existing variables: metric_range, trace_range, log_range, error_range, metric_agg, trace_agg, log_agg

min_metric = metric_range.loc[0, 'min_ts']
max_metric = metric_range.loc[0, 'max_ts']
min_trace = trace_range.loc[0, 'min_ts']
max_trace = trace_range.loc[0, 'max_ts']
min_log = log_range.loc[0, 'min_ts']
max_log = log_range.loc[0, 'max_ts']

# Pick a few representative top groups from the already-computed aggregates for examples
top_metric_examples = metric_agg.head(5).to_dict(orient='records')
top_trace_examples = trace_agg.head(5).to_dict(orient='records')
top_log_examples = log_agg.head(5).to_dict(orient='records']

summary = (
    "Summary of telemetry files (UTC):\n\n"
    f"Timestamp ranges:\n"
    f"- metric.csv: {min_metric} to {max_metric}\n"
    f"- trace.csv:  {min_trace} to {max_trace}\n"
    f"- log.csv:    {min_log} to {max_log}\n"
    f"- error_logs.csv: no rows found (empty)\n\n"
    "High-level findings:\n"
    "- Data covers a ~25-minute window with most metric/trace/log groups having 25 samples (some 24).\n"
    "- No error log rows were present in error_logs.csv.\n\n"
    "Representative top metric groups (group, count, p95):\n"
    + "\n".join([f"  - {r['cmdb_id']} / {r['kpi_name']}: count={r['count']}, p95={r['p95']}" for r in top_metric_examples])
    + "\n\nRepresentative top trace groups (group, count, p95):\n"
    + "\n".join([f"  - {r['cmdb_id']} / {r['trace_name']}: count={r['count']}, p95={r['p95']}" for r in top_trace_examples])
    + "\n\nRepresentative top log groups (group, count, p95):\n"
    + "\n".join([f"  - {r['cmdb_id']} / {r['log_name']}: count={r['count']}, p95={r['p95']}" for r in top_log_examples])
    + "\n\nInterpretation and next steps:\n"
    "- Logs show low error_count p95 (0) for most services, so few logged errors in this window.\n"
    "- Metrics indicate some services with large diskio and memory p95 values and occasional high socket counts; traces show some p95 durations up to a few seconds for cross-service calls.\n"
    "- To investigate root cause, focus on services with high metric p95 (diskio/mem/socket) and the trace entries showing high duration p95 (examples listed above).\n"
    "- If deeper investigation is needed, request expanded slices (longer time window or raw rows for the specific groups listed).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file                    min_ts                    max_ts
0  metric.csv 2024-01-23 10:59:00+00:00 2024-01-23 11:23:00+00:00,                          cmdb_id    kpi_name  count           p95
355           ts-voucher-service      socket     25  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  2.044671e+00
1    ts-admin-basic-info-service      diskio     25  1.896397e+06
2    ts-admin-basic-info-service  latency-50     25  2.220588e-02
316             ts-travel2-mongo      diskio     25  4.993707e+05
..                           ...         ...    ...           ...
53              ts-basic-service      diskio     25  2.127274e+06
54              ts-basic-service  latency-50     25  1.917806e-02
55              ts-basic-service  latency-90     25  1.441056e-01
56              ts-basic-service         mem     25  2.385078e+08
57              ts-basic-service      socket     25  1.544333e+01

[50 rows x 4 columns],         file                    min_ts                    max_ts
0  trace.csv 2024-01-23 10:59:00+00:00 2024-01-23 11:23:00+00:00,                          cmdb_id                                         trace_name  count        p95
775              ts-user-service                 trace.to_ts-user-service.row_count     25  20.000000
0                           root  trace.to_ts-admin-basic-info-service.duration_...     25   0.022078
1                           root  trace.to_ts-admin-basic-info-service.duration_p95     25   0.048727
2                           root    trace.to_ts-admin-basic-info-service.error_rate     25   0.000000
3                           root     trace.to_ts-admin-basic-info-service.row_count     25  20.000000
..                           ...                                                ...    ...        ...
61   ts-admin-basic-info-service                       trace.from_root.duration_p95     25   0.048727
62   ts-admin-basic-info-service                         trace.from_root.error_rate     25   0.000000
63   ts-admin-basic-info-service                          trace.from_root.row_count     25  20.000000
52                          root           trace.to_ts-travel-service.duration_mean     25   2.152329
53                          root            trace.to_ts-travel-service.duration_p95     25   3.515914

[50 rows x 4 columns],       file                    min_ts                    max_ts
0  log.csv 2024-01-23 10:59:00+00:00 2024-01-23 11:23:00+00:00,                         cmdb_id         log_name  count    p95
0   ts-admin-basic-info-service  log.error_count     25   0.00
1   ts-admin-basic-info-service    log.row_count     25  29.80
2       ts-admin-travel-service  log.error_count     25   0.00
3       ts-admin-travel-service    log.row_count     25  32.40
7          ts-assurance-service    log.row_count     25  16.60
..                          ...              ...    ...    ...
8                 ts-auth-mongo  log.error_count     24   0.00
4            ts-assurance-mongo  log.error_count     24   0.00
5            ts-assurance-mongo    log.row_count     24   1.00
29            ts-food-map-mongo    log.row_count     24   1.85
28            ts-food-map-mongo  log.error_count     24   0.00

[50 rows x 4 columns],              file min_ts max_ts
0  error_logs.csv    NaT    NaT, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse df_metric from prior state; ensure timestamp is parsed and UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series BEFORE any filtering
global_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

total_groups_scanned = global_p95.shape[0]

# 2) Filter metric.csv rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 10:56:52", utc=True)
end = pd.to_datetime("2024-01-23 11:26:52", utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# 3) Merge windowed rows with global_p95 and mark anomalies (value >= global_p95)
df_win = df_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
df_win['anomaly'] = df_win['value'] >= df_win['global_p95']

# Aggregations within window
agg_window = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        window_count=('value', 'count'),
        anomaly_count=('anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest/latest anomaly timestamps (from anomaly rows only)
df_anom = df_win[df_win['anomaly']].copy()
anom_times = (
    df_anom
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        earliest_anomaly_ts=('timestamp', 'min'),
        latest_anomaly_ts=('timestamp', 'max')
    )
)

# Combine results and include global_p95
results = (
    agg_window
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anom_times, on=['cmdb_id', 'kpi_name'], how='left')
)

# 4) Compute severity_pct; handle global_p95 == 0 by setting severity_pct = NaN
results['severity_pct'] = np.where(
    results['global_p95'] == 0,
    np.nan,
    (results['max_value_in_window'] - results['global_p95']) / results['global_p95'] * 100.0
)

# 5) Filter to groups with anomaly_count > 0, format timestamps to ISO UTC strings, select columns and sort
results_filtered = results[results['anomaly_count'] > 0].copy()

# Format timestamps as ISO UTC strings
results_filtered['earliest_anomaly_ts'] = results_filtered['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
results_filtered['latest_anomaly_ts'] = results_filtered['latest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order columns as requested
final_cols = [
    'cmdb_id', 'kpi_name', 'window_count', 'anomaly_count',
    'earliest_anomaly_ts', 'latest_anomaly_ts',
    'global_p95', 'max_value_in_window', 'severity_pct'
]
final_table = (
    results_filtered[final_cols]
    .sort_values(['anomaly_count', 'severity_pct'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# 6) Summary numbers
total_anomaly_rows = int(df_win['anomaly'].sum())

# Display compact outputs: final_table, total_groups_scanned, total_anomaly_rows
final_table, total_groups_scanned, total_anomaly_rows

```
Out[2]:
```
```python
# Compose a plain-English summary string and show top 5 groups for context.
window_start = "2024-01-23 10:56:52 UTC"
window_end = "2024-01-23 11:26:52 UTC"

summary = (
    f"Summary for metric anomalies in window {window_start} to {window_end}:\n\n"
    f"- Total metric groups scanned (global P95 computed on full series): {total_groups_scanned}\n"
    f"- Total anomaly rows in the window (value >= global P95): {total_anomaly_rows}\n\n"
    "Key observations:\n"
    "- Many of the detected anomaly groups are 'socket' metrics across multiple services (see examples below).\n"
    "- For the top groups, anomaly_count equals the full window_count (25), but severity_pct is 0.0 for these rows. "
    "This means values reached the group's global P95 but did not exceed it (max_value_in_window == global_p95).\n\n"
    "Interpretation and suggested next steps:\n"
    "- The alerting criterion used here is value >= global_p95; counts therefore include values exactly at the P95. "
    "If you want to detect values exceeding expected ranges, consider using value > global_p95.\n"
    "- Focus investigation on services with high socket counts or large memory/diskio p95 values, but note that "
    "a severity_pct of 0 indicates no exceedance beyond the global P95 in this window.\n"
    "- If deeper analysis is needed, inspect raw time series for the listed groups to understand trends and spikes.\n"
)

# Show the summary and the top 5 anomalous groups (compact table) for context
summary, final_table.head(5)
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id kpi_name  window_count  anomaly_count   earliest_anomaly_ts     latest_anomaly_ts   global_p95  max_value_in_window  severity_pct
0              ts-auth-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          5.0                  5.0           0.0
1          ts-avatar-service   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          5.0                  5.0           0.0
2           ts-consign-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          4.0                  4.0           0.0
3     ts-consign-price-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          4.0                  4.0           0.0
4              ts-food-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          5.0                  5.0           0.0
5              ts-food-mysql      mem            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z  599855104.0          599855104.0           0.0
6              ts-food-mysql   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          2.0                  2.0           0.0
7    ts-inside-payment-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          4.0                  4.0           0.0
8            ts-news-service   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          1.0                  1.0           0.0
9      ts-notification-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          2.0                  2.0           0.0
10      ts-order-other-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z         18.0                 18.0           0.0
11          ts-payment-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          4.0                  4.0           0.0
12         ts-preserve-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          2.0                  2.0           0.0
13   ts-preserve-other-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          2.0                  2.0           0.0
14            ts-price-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z         20.0                 20.0           0.0
15            ts-route-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          8.0                  8.0           0.0
16         ts-security-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          6.0                  6.0           0.0
17          ts-station-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z         18.0                 18.0           0.0
18    ts-ticket-office-mongo   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          2.0                  2.0           0.0
19  ts-ticket-office-service   socket            25             25  2024-01-23T10:59:00Z  2024-01-23T11:23:00Z          1.0                  1.0           0.0, 356, 1540)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse and ensure base variables
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure global_p95 is computed (computed earlier, but recompute to be safe and to follow rule 9)
global_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)
total_groups_scanned = global_p95.shape[0]

# Incident window (UTC)
start = pd.to_datetime("2024-01-23 10:56:52", utc=True)
end = pd.to_datetime("2024-01-23 11:26:52", utc=True)

# Filter rows to the window
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# Merge with global_p95 and mark strict exceedances (value > global_p95)
df_win = df_window.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
df_win['strict_anomaly'] = df_win['value'] > df_win['global_p95']

# Aggregations within window
agg_window = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        window_count=('value', 'count'),
        strict_anomaly_count=('strict_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest/latest strict anomaly timestamps (from strict anomaly rows only)
df_strict = df_win[df_win['strict_anomaly']].copy()
strict_times = (
    df_strict
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        earliest_strict_anomaly_ts=('timestamp', 'min'),
        latest_strict_anomaly_ts=('timestamp', 'max')
    )
)

# Combine results and include global_p95
results_strict = (
    agg_window
    .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(strict_times, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute severity_pct, handle global_p95 == 0
results_strict['severity_pct'] = np.where(
    results_strict['global_p95'] == 0,
    np.nan,
    (results_strict['max_value_in_window'] - results_strict['global_p95']) / results_strict['global_p95'] * 100.0
)

# Filter to groups with at least one strict exceedance
results_strict = results_strict[results_strict['strict_anomaly_count'] > 0].copy()

# Format timestamps as ISO UTC strings
results_strict['earliest_strict_anomaly_ts'] = results_strict['earliest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
results_strict['latest_strict_anomaly_ts'] = results_strict['latest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order columns, sort by strict_anomaly_count desc then severity_pct desc, limit to top 20
final_cols_strict = [
    'cmdb_id', 'kpi_name', 'window_count', 'strict_anomaly_count',
    'earliest_strict_anomaly_ts', 'latest_strict_anomaly_ts',
    'global_p95', 'max_value_in_window', 'severity_pct'
]
final_table_strict = (
    results_strict[final_cols_strict]
    .sort_values(['strict_anomaly_count', 'severity_pct'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# Summary numbers
total_strict_anomaly_rows = int(df_win['strict_anomaly'].sum())

# Return compact outputs
final_table_strict, total_groups_scanned, total_strict_anomaly_rows

```
Out[3]:
```
```python
# Build a concise plain-English summary using existing results in the kernel:
rows = final_table_strict.head(5).to_dict(orient='records')

summary_lines = [
    f"Summary of strict metric exceedances (value > global P95) in window 2024-01-23 10:56:52 to 2024-01-23 11:26:52 UTC:",
    f"- Total metric groups scanned (global P95 computed on full series): {total_groups_scanned}",
    f"- Total strict-exceedance rows found in the window: {total_strict_anomaly_rows}",
    "",
    "Top example groups with strict exceedances (showing up to 5):"
]

for r in rows:
    summary_lines.append(
        f"  - {r['cmdb_id']} / {r['kpi_name']}: "
        f"strict_anomaly_count={int(r['strict_anomaly_count'])}, "
        f"window_count={int(r['window_count'])}, "
        f"earliest={r['earliest_strict_anomaly_ts']}, latest={r['latest_strict_anomaly_ts']}, "
        f"global_p95={r['global_p95']:.6g}, max_in_window={r['max_value_in_window']:.6g}, "
        f"severity_pct={r['severity_pct']:.2f}%"
    )

summary_lines += [
    "",
    "Key observations:",
    "- Strict exceedances are concentrated in latency KPIs (latency-90 and latency-50) across multiple services.",
    "- The largest relative severities are very high (e.g., thousands of percent) because global P95 values for some latency KPIs are very small and peak latencies are orders of magnitude larger.",
    "",
    "Recommended next steps:",
    "- Investigate the listed services' request traces and logs around the reported timestamps for root-cause (slow downstream calls, retries, or resource contention).",
    "- Correlate with infrastructure metrics (CPU, memory, disk I/O, socket counts) and any recent deployments or config changes."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id    kpi_name  window_count  strict_anomaly_count earliest_strict_anomaly_ts latest_strict_anomaly_ts  global_p95  max_value_in_window  severity_pct
0    ts-consign-price-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:01:00Z    0.062715             1.482976   2264.616984
1        ts-assurance-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.160301             2.994543   1768.080105
2          ts-payment-service  latency-90            24                     2       2024-01-23T11:00:00Z     2024-01-23T11:20:00Z    0.161554             2.772167   1615.941158
3         ts-security-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.284386             3.226402   1034.514636
4          ts-consign-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.715380             6.824603    853.982733
5         ts-contacts-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.596530             3.705100    521.109156
6   ts-preserve-other-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    2.793829            16.250000    481.638986
7         ts-preserve-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    2.757030            15.000000    444.063662
8   ts-preserve-other-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    1.313487             5.998942    356.718926
9         ts-contacts-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.030253             0.136816    352.244953
10  ts-inside-payment-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    1.930696             8.019025    315.343815
11  ts-inside-payment-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:12:00Z    1.518495             6.136234    304.099808
12        ts-preserve-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    1.526022             5.620621    268.318530
13           ts-basic-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.019178             0.067951    254.315886
14        ts-security-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    1.315248             4.645280    253.186582
15            ts-user-service  latency-90            24                     2       2024-01-23T11:02:00Z     2024-01-23T11:03:00Z    0.039317             0.134297    241.573796
16          ts-travel-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.051954             0.162999    213.734420
17         ts-consign-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:11:00Z    1.557797             4.404056    182.710477
18      ts-ticketinfo-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.032678             0.090944    178.303941
19     ts-order-other-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.035608             0.092477    159.710391, 356, 629)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Ensure df_trace is available and timestamp parsed as UTC (reuse existing variable if present)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (before filtering)
trace_global_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)
total_trace_groups_scanned = trace_global_p95.shape[0]

# 2) Filter trace rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 10:56:52", utc=True)
end = pd.to_datetime("2024-01-23 11:26:52", utc=True)
df_trace_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# 3) Merge with global_p95 and mark strict exceedances (value > global_p95)
df_tw = df_trace_window.merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')
df_tw['strict_anomaly'] = df_tw['value'] > df_tw['global_p95']

# Aggregations within the window
agg_trace_window = (
    df_tw
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        window_count=('value', 'count'),
        strict_anomaly_count=('strict_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest/latest strict anomaly timestamps (from strict anomaly rows only)
df_trace_strict = df_tw[df_tw['strict_anomaly']].copy()
strict_times = (
    df_trace_strict
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        earliest_strict_anomaly_ts=('timestamp', 'min'),
        latest_strict_anomaly_ts=('timestamp', 'max')
    )
)

# Combine results and include global_p95
results_trace = (
    agg_trace_window
    .merge(trace_global_p95, on=['cmdb_id', 'trace_name'], how='left')
    .merge(strict_times, on=['cmdb_id', 'trace_name'], how='left')
)

# 3-4) Compute severity_pct; handle global_p95 == 0 by setting NaN
results_trace['severity_pct'] = np.where(
    results_trace['global_p95'] == 0,
    np.nan,
    (results_trace['max_value_in_window'] - results_trace['global_p95']) / results_trace['global_p95'] * 100.0
)

# Filter to groups with at least one strict exceedance
results_trace = results_trace[results_trace['strict_anomaly_count'] > 0].copy()

# Format timestamps as ISO UTC strings
results_trace['earliest_strict_anomaly_ts'] = results_trace['earliest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
results_trace['latest_strict_anomaly_ts'] = results_trace['latest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order columns, sort by strict_anomaly_count desc then severity_pct desc, limit to top 20
final_cols_trace = [
    'cmdb_id', 'trace_name', 'window_count', 'strict_anomaly_count',
    'earliest_strict_anomaly_ts', 'latest_strict_anomaly_ts',
    'global_p95', 'max_value_in_window', 'severity_pct'
]
final_table_trace = (
    results_trace[final_cols_trace]
    .sort_values(['strict_anomaly_count', 'severity_pct'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# 5) Summary counts: total groups scanned and total strict-exceedance trace rows in the window
total_strict_exceed_rows_trace = int(df_tw['strict_anomaly'].sum())

# Return compact outputs
final_table_trace, total_trace_groups_scanned, total_strict_exceed_rows_trace

```
Out[4]:
```
```python
# Compose a plain-English summary of trace anomaly analysis using existing results in the kernel:
rows = final_table_trace.head(5).to_dict(orient='records')

summary_lines = [
    "Trace-level anomaly summary (strict exceedance: value > global P95) for window 2024-01-23 10:56:52 to 2024-01-23 11:26:52 UTC:",
    f"- Total trace groups scanned (global P95 computed on full series): {total_trace_groups_scanned}",
    f"- Total strict-exceedance trace rows found in the window: {total_strict_exceed_rows_trace}",
    "",
    "Top example trace groups with strict exceedances (up to 5):"
]

for r in rows:
    summary_lines.append(
        f"  - {r['cmdb_id']} / {r['trace_name']}: "
        f"strict_anomaly_count={int(r['strict_anomaly_count'])}, window_count={int(r['window_count'])}, "
        f"earliest={r['earliest_strict_anomaly_ts']}, latest={r['latest_strict_anomaly_ts']}, "
        f"global_p95={r['global_p95']:.6g}, max_in_window={r['max_value_in_window']:.6g}, "
        f"severity_pct={r['severity_pct']:.2f}%"
    )

summary_lines += [
    "",
    "Key findings:",
    "- Strict exceedances are concentrated in trace duration metrics (duration_mean and duration_p95) across multiple services.",
    "- The highest-impact groups include ts-user-service (incoming and outgoing durations), ts-consign-price-service / ts-consign-service, ts-assurance-service, ts-route-service / ts-travel2-service, ts-contacts-service, and ts-admin-basic-info-service.",
    "- Reported severity_pct values are very large for many groups because their baseline global P95 durations are very small while observed peak durations are substantially larger.",
    "",
    "Practical next steps to localize the fault:",
    "- Inspect full traces (spans) for the listed services and timestamps to identify slow downstream calls or hotspots.",
    "- Correlate with service logs and infrastructure metrics (CPU, memory, disk I/O, network/socket) around the earliest_strict_anomaly_ts for each service.",
    "- Check for recent deployments, configuration changes, or increased traffic that could explain simultaneous latency spikes.",
    "",
    "Conclusion:",
    "- Trace telemetry points to application-level latency issues affecting multiple services, with ts-user-service and various consign/assurance/route-related services among the top suspects. Focus trace/span analysis and logs on those services for root-cause."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id                                         trace_name  window_count  strict_anomaly_count earliest_strict_anomaly_ts latest_strict_anomaly_ts  global_p95  max_value_in_window  severity_pct
0               ts-user-service            trace.from_ts-user-service.duration_p95            25                     2       2024-01-23T11:02:00Z     2024-01-23T11:17:00Z    0.005333             0.048107    802.020160
1               ts-user-service              trace.to_ts-user-service.duration_p95            25                     2       2024-01-23T11:02:00Z     2024-01-23T11:17:00Z    0.005333             0.048107    802.020160
2      ts-consign-price-service        trace.from_ts-consign-service.duration_mean            24                     2       2024-01-23T11:00:00Z     2024-01-23T11:12:00Z    0.014406             0.067860    371.059493
3            ts-consign-service    trace.to_ts-consign-price-service.duration_mean            24                     2       2024-01-23T11:00:00Z     2024-01-23T11:12:00Z    0.014406             0.067860    371.059493
4              ts-route-service         trace.from_ts-travel2-service.duration_p95            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.011369             0.051049    349.017692
5            ts-travel2-service             trace.to_ts-route-service.duration_p95            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.011369             0.051049    349.017692
6          ts-assurance-service      trace.from_ts-assurance-service.duration_mean            25                     2       2024-01-23T11:05:00Z     2024-01-23T11:09:00Z    0.006397             0.026638    316.400276
7          ts-assurance-service        trace.to_ts-assurance-service.duration_mean            25                     2       2024-01-23T11:05:00Z     2024-01-23T11:09:00Z    0.006397             0.026638    316.400276
8          ts-assurance-service       trace.from_ts-assurance-service.duration_p95            25                     2       2024-01-23T11:05:00Z     2024-01-23T11:09:00Z    0.019571             0.075200    284.242951
9          ts-assurance-service         trace.to_ts-assurance-service.duration_p95            25                     2       2024-01-23T11:05:00Z     2024-01-23T11:09:00Z    0.019571             0.075200    284.242951
10              ts-user-service           trace.from_ts-user-service.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:02:00Z    0.003193             0.010795    238.119401
11              ts-user-service             trace.to_ts-user-service.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:02:00Z    0.003193             0.010795    238.119401
12                         root         trace.to_ts-contacts-service.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:18:00Z    0.029806             0.094794    218.037907
13          ts-contacts-service                      trace.from_root.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:18:00Z    0.029806             0.094794    218.037907
14  ts-admin-basic-info-service  trace.from_ts-admin-basic-info-service.duratio...            25                     2       2024-01-23T11:20:00Z     2024-01-23T11:22:00Z    0.033088             0.102704    210.397305
15  ts-admin-basic-info-service  trace.to_ts-admin-basic-info-service.duration_p95            25                     2       2024-01-23T11:20:00Z     2024-01-23T11:22:00Z    0.033088             0.102704    210.397305
16    ts-inside-payment-service       trace.to_ts-order-other-service.duration_p95            23                     2       2024-01-23T11:18:00Z     2024-01-23T11:20:00Z    0.036767             0.106711    190.239664
17       ts-order-other-service  trace.from_ts-inside-payment-service.duration_p95            23                     2       2024-01-23T11:18:00Z     2024-01-23T11:20:00Z    0.036767             0.106711    190.239664
18     ts-consign-price-service         trace.from_ts-consign-service.duration_p95            24                     2       2024-01-23T11:00:00Z     2024-01-23T11:12:00Z    0.036706             0.106383    189.821433
19           ts-consign-service     trace.to_ts-consign-price-service.duration_p95            24                     2       2024-01-23T11:00:00Z     2024-01-23T11:12:00Z    0.036706             0.106383    189.821433, 776, 1064)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_metric (assumed loaded previously) and ensure timestamp parsed as UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Components of interest
components = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# 1) Compute global P95 for each (cmdb_id, kpi_name) restricted to the selected components, using the full series
df_subset_full = df_metric[df_metric['cmdb_id'].isin(components)].copy()
global_p95_subset = (
    df_subset_full
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)
total_groups_scanned_subset = global_p95_subset.shape[0]

# 2) Filter rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 10:56:52", utc=True)
end = pd.to_datetime("2024-01-23 11:26:52", utc=True)
df_window_subset = df_subset_full[(df_subset_full['timestamp'] >= start) & (df_subset_full['timestamp'] <= end)].copy()

# 3) Merge with global_p95 and mark strict exceedances (value > global_p95)
df_ws = df_window_subset.merge(global_p95_subset, on=['cmdb_id', 'kpi_name'], how='left')
df_ws['strict_anomaly'] = df_ws['value'] > df_ws['global_p95']

# Aggregations within window
agg_window_subset = (
    df_ws
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        window_count=('value', 'count'),
        strict_anomaly_count=('strict_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest/latest strict anomaly timestamps (from strict anomaly rows only)
df_strict_subset = df_ws[df_ws['strict_anomaly']].copy()
strict_times_subset = (
    df_strict_subset
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        earliest_strict_anomaly_ts=('timestamp', 'min'),
        latest_strict_anomaly_ts=('timestamp', 'max')
    )
)

# Combine results and include global_p95
results_subset = (
    agg_window_subset
    .merge(global_p95_subset, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(strict_times_subset, on=['cmdb_id', 'kpi_name'], how='left')
)

# 4) Compute severity_pct; handle global_p95 == 0
results_subset['severity_pct'] = np.where(
    results_subset['global_p95'] == 0,
    np.nan,
    (results_subset['max_value_in_window'] - results_subset['global_p95']) / results_subset['global_p95'] * 100.0
)

# Filter to groups with at least one strict exceedance
results_strict_subset = results_subset[results_subset['strict_anomaly_count'] > 0].copy()

# Format timestamps as ISO UTC strings
results_strict_subset['earliest_strict_anomaly_ts'] = results_strict_subset['earliest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
results_strict_subset['latest_strict_anomaly_ts'] = results_strict_subset['latest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order columns, sort by strict_anomaly_count desc then severity_pct desc, limit to top 20
final_cols = [
    'cmdb_id', 'kpi_name', 'window_count', 'strict_anomaly_count',
    'earliest_strict_anomaly_ts', 'latest_strict_anomaly_ts',
    'global_p95', 'max_value_in_window', 'severity_pct'
]
final_table_subset = (
    results_strict_subset[final_cols]
    .sort_values(['strict_anomaly_count', 'severity_pct'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# 5) Summary: total strict-exceedance rows found in the window for these components
total_strict_exceed_rows_subset = int(df_ws['strict_anomaly'].sum())

# Return compact outputs
final_table_subset, total_groups_scanned_subset, total_strict_exceed_rows_subset

```
Out[5]:
```
Summary (plain English)

- Scope and counts:
  - Components scanned: 36 (only the five requested services).
  - Total strict-exceedance metric rows in the incident window (value > global P95): 70.

- What was found (high-level):
  - Strict exceedances occur across multiple metric types, but the most common/impactful are latency metrics (latency-50 and latency-90), CPU, disk I/O, workload, memory and socket counts.
  - Every reported group with strict exceedances has 2 strict-exceedance points in the window (strict_anomaly_count = 2).
  - Anomalies appear in the incident window roughly between 2024-01-23T10:59:00Z and 2024-01-23T11:23:00Z (earliest/latest anomaly timestamps fall in that range).

- Top affected services & metrics (recommendation priority):
  1. ts-travel-service — latency-50 and latency-90, CPU, workload, socket, diskio (notable high relative severity; latency-50 severity ≈ +214%).
  2. ts-route-service — latency-50 and latency-90, CPU, socket (high relative latency severity).
  3. ts-order-service — diskio, latency-90, CPU, workload (significant disk I/O and CPU increases).
  4. ts-auth-service — CPU and latency-50/90 (moderate severity).
  5. ts-train-service — workload and memory (moderate severity).

- Interpretation:
  - The telemetry indicates application-level latency spikes concentrated on ts-travel, ts-route and ts-order (plus related CPU/diskio increases for order). These services should be the first focus for root-cause analysis.
  - Many latency severity_pct values are large because baseline P95 latencies are small and peaks are substantially larger — this signals real, meaningful slowdowns, not just tiny fluctuations.

- Recommended next steps:
  1. Inspect traces/spans and service logs for ts-travel-service, ts-route-service, and ts-order-service at the reported timestamps (around 10:59–11:23 UTC) to find slow downstream calls, retries, or errors.
  2. Correlate with infra metrics (host/container CPU, memory, disk I/O, network/socket statistics) for the same services and times — ts-order-service shows notable disk I/O and CPU increases.
  3. Check for recent deployments or configuration changes and any traffic surge that could explain simultaneous spikes.
  4. If needed, expand the time window slightly and fetch raw series for the top groups to validate patterns and identify exact span/operation causing the latency.

If you want, I can produce the compact table of the top 20 groups (with the columns you requested) or expand the window or include related trace/log correlations next.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  window_count  strict_anomaly_count earliest_strict_anomaly_ts latest_strict_anomaly_ts    global_p95  max_value_in_window  severity_pct
0   ts-travel-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  5.195432e-02         1.629986e-01    213.734420
1    ts-route-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  7.966885e-03         2.028683e-02    154.639373
2    ts-route-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  3.851501e-02         9.440961e-02    145.124226
3    ts-order-service      diskio            25                     2       2024-01-23T11:13:00Z     2024-01-23T11:19:00Z  5.803177e+04         1.034862e+05     78.326873
4    ts-order-service  latency-90            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:21:00Z  6.163378e-02         1.095880e-01     77.805063
5    ts-order-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  1.079508e+01         1.855194e+01     71.855433
6   ts-travel-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  1.643920e+01         2.816089e+01     71.303303
7    ts-route-service         cpu            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  1.665627e+01         2.765643e+01     66.042186
8    ts-order-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  1.362719e-02         2.077350e-02     52.441551
9     ts-auth-service         cpu            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:03:00Z  1.779374e+01         2.129301e+01     19.665707
10  ts-travel-service    workload            25                     2       2024-01-23T11:11:00Z     2024-01-23T11:15:00Z  1.039518e+01         1.214447e+01     16.827826
11   ts-order-service    workload            25                     2       2024-01-23T11:11:00Z     2024-01-23T11:15:00Z  7.332850e+00         8.194517e+00     11.750774
12    ts-auth-service  latency-50            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z  3.833789e-01         4.221705e-01     10.118349
13   ts-route-service      socket            25                     2       2024-01-23T11:16:00Z     2024-01-23T11:23:00Z  1.558635e+01         1.703333e+01      9.283642
14   ts-train-service    workload            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:11:00Z  2.163877e+01         2.357997e+01      8.970920
15    ts-auth-service  latency-90            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:03:00Z  1.240581e+00         1.331033e+00      7.291126
16  ts-travel-service      socket            25                     2       2024-01-23T11:13:00Z     2024-01-23T11:18:00Z  2.064333e+01         2.201667e+01      6.652672
17  ts-travel-service  latency-90            25                     2       2024-01-23T11:17:00Z     2024-01-23T11:21:00Z  1.143396e+00         1.213704e+00      6.149078
18   ts-train-service         mem            25                     2       2024-01-23T11:18:00Z     2024-01-23T11:22:00Z  1.013252e+09         1.073737e+09      5.969353
19  ts-travel-service      diskio            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:19:00Z  5.967685e+04         6.304956e+04      5.651614, 36, 70)```
```

In [ ]:
# In[6]:

import re
import numpy as np
import pandas as pd

# Reuse existing df_trace and ensure timestamp parsed as UTC
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Build patterns: 'to_ts-{name}', 'from_ts-{name}', and the service name itself
patterns = []
for s in candidates:
    patterns.extend([f"to_ts-{s}", f"from_ts-{s}", s])

# Compile regex that matches any of the patterns (escape to be safe)
regex = "|".join(re.escape(p) for p in patterns)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace series (before filtering)
trace_global_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Select trace groups where trace_name contains any candidate-related token (case-sensitive substring match)
mask_selected = trace_global_p95['trace_name'].str.contains(regex, regex=True)
trace_global_p95_sel = trace_global_p95[mask_selected].copy()
total_trace_groups_scanned_selected = trace_global_p95_sel.shape[0]

# 2) Filter trace rows to the incident window (UTC)
start = pd.to_datetime("2024-01-23 10:56:52", utc=True)
end = pd.to_datetime("2024-01-23 11:26:52", utc=True)
df_trace_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# Further restrict window rows to those trace_name that are in our selected set
selected_names = trace_global_p95_sel['trace_name'].unique()
df_tw_sel = df_trace_window[df_trace_window['trace_name'].isin(selected_names)].copy()

# 3) Merge with global_p95 (selected) and mark strict exceedances (value > global_p95)
df_tw_sel = df_tw_sel.merge(trace_global_p95_sel, on=['cmdb_id', 'trace_name'], how='left')
df_tw_sel['strict_anomaly'] = df_tw_sel['value'] > df_tw_sel['global_p95']

# Aggregations within the window
agg_trace_sel = (
    df_tw_sel
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        window_count=('value', 'count'),
        strict_anomaly_count=('strict_anomaly', 'sum'),
        max_value_in_window=('value', 'max')
    )
)

# earliest/latest strict anomaly timestamps (from strict anomaly rows only)
df_trace_strict_sel = df_tw_sel[df_tw_sel['strict_anomaly']].copy()
strict_times_sel = (
    df_trace_strict_sel
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        earliest_strict_anomaly_ts=('timestamp', 'min'),
        latest_strict_anomaly_ts=('timestamp', 'max')
    )
)

# Combine results and include global_p95
results_trace_sel = (
    agg_trace_sel
    .merge(trace_global_p95_sel, on=['cmdb_id', 'trace_name'], how='left')
    .merge(strict_times_sel, on=['cmdb_id', 'trace_name'], how='left')
)

# 3-4) Compute severity_pct; handle global_p95 == 0 by setting NaN
results_trace_sel['severity_pct'] = np.where(
    results_trace_sel['global_p95'] == 0,
    np.nan,
    (results_trace_sel['max_value_in_window'] - results_trace_sel['global_p95']) / results_trace_sel['global_p95'] * 100.0
)

# Filter to groups with at least one strict exceedance
results_trace_sel = results_trace_sel[results_trace_sel['strict_anomaly_count'] > 0].copy()

# Format timestamps as ISO UTC strings
results_trace_sel['earliest_strict_anomaly_ts'] = results_trace_sel['earliest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
results_trace_sel['latest_strict_anomaly_ts'] = results_trace_sel['latest_strict_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select and order columns, sort by strict_anomaly_count desc then severity_pct desc, limit to top 20
final_cols = [
    'cmdb_id', 'trace_name', 'window_count', 'strict_anomaly_count',
    'earliest_strict_anomaly_ts', 'latest_strict_anomaly_ts',
    'global_p95', 'max_value_in_window', 'severity_pct'
]
final_table_trace_selected = (
    results_trace_sel[final_cols]
    .sort_values(['strict_anomaly_count', 'severity_pct'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# 5) Summary numbers
total_strict_exceed_rows_selected = int(df_tw_sel['strict_anomaly'].sum())

# Return compact outputs
final_table_trace_selected, total_trace_groups_scanned_selected, total_strict_exceed_rows_selected

```
Out[6]:
```
```python
# Compose a plain-English summary using existing results in the kernel
rows = final_table_trace_selected.head(5).to_dict(orient='records')

summary_lines = [
    "Trace-analysis summary for candidate services (strict exceedance = value > global P95):",
    f"- Total candidate-related trace groups scanned: {total_trace_groups_scanned_selected}",
    f"- Total strict-exceedance trace rows found in the incident window: {total_strict_exceed_rows_selected}",
    "",
    "Top example trace groups with strict exceedances (up to 5):"
]

for r in rows:
    summary_lines.append(
        f"  - {r['cmdb_id']} / {r['trace_name']}: "
        f"strict_anomaly_count={int(r['strict_anomaly_count'])}, window_count={int(r['window_count'])}, "
        f"earliest={r['earliest_strict_anomaly_ts']}, latest={r['latest_strict_anomaly_ts']}, "
        f"global_p95={r['global_p95']:.6g}, max_in_window={r['max_value_in_window']:.6g}, "
        f"severity_pct={r['severity_pct']:.2f}%"
    )

summary_lines += [
    "",
    "Key findings:",
    "- Significant strict exceedances are seen on traces involving route and auth interactions, notably:",
    "  * ts-travel2-service -> trace.to_ts-route-service.duration_p95 (very large relative spike)",
    "  * ts-auth-service (both incoming and outgoing duration_p95 traces)",
    "  * ts-order-service and ts-travel-service traces connecting to each other",
    "- Many exceedances cluster around 2024-01-23T11:03–11:05Z and some extend later (e.g., auth traces up to ~11:22Z).",
    "- High severity_pct values are driven by small baseline P95 durations with much larger observed peaks, indicating meaningful latency spikes.",
    "",
    "Recommended next steps:",
    "- Inspect full spans/traces for the listed trace groups and timestamps to find slow downstream calls, retries, or blocked resources.",
    "- Correlate with service logs and infrastructure metrics (CPU, memory, disk I/O, sockets) for ts-travel2-service, ts-travel-service, ts-auth-service, and ts-order-service around the reported times.",
    "- Prioritize investigation of route-related calls and auth interactions, as they appear repeatedly among top anomalies."
]

summary = "\n".join(summary_lines)
summary, final_table_trace_selected.head(5)
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                  trace_name  window_count  strict_anomaly_count earliest_strict_anomaly_ts latest_strict_anomaly_ts  global_p95  max_value_in_window  severity_pct
0      ts-travel2-service      trace.to_ts-route-service.duration_p95            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.011369             0.051049    349.017692
1         ts-auth-service     trace.from_ts-auth-service.duration_p95            25                     2       2024-01-23T11:02:00Z     2024-01-23T11:22:00Z    0.032863             0.085576    160.402847
2         ts-auth-service       trace.to_ts-auth-service.duration_p95            25                     2       2024-01-23T11:02:00Z     2024-01-23T11:22:00Z    0.032863             0.085576    160.402847
3        ts-order-service   trace.from_ts-travel-service.duration_p95            25                     2       2024-01-23T11:03:00Z     2024-01-23T11:05:00Z    0.015792             0.040462    156.214824
4       ts-travel-service      trace.to_ts-order-service.duration_p95            25                     2       2024-01-23T11:03:00Z     2024-01-23T11:05:00Z    0.015792             0.040462    156.214824
5     ts-security-service      trace.to_ts-order-service.duration_p95            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:05:00Z    0.063680             0.133828    110.156465
6                    root       trace.to_ts-auth-service.duration_p95            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:02:00Z    1.121700             2.165678     93.071123
7        ts-route-service   trace.from_ts-travel-service.duration_p95            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.033132             0.060345     82.136303
8       ts-travel-service      trace.to_ts-route-service.duration_p95            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.033132             0.060345     82.136303
9   ts-ticketinfo-service   trace.from_ts-travel-service.duration_p95            25                     2       2024-01-23T11:14:00Z     2024-01-23T11:22:00Z    0.371804             0.634314     70.604705
10        ts-seat-service      trace.to_ts-order-service.duration_p95            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:12:00Z    0.043463             0.067276     54.786200
11     ts-station-service    trace.from_ts-order-service.duration_p95            24                     2       2024-01-23T11:22:00Z     2024-01-23T11:23:00Z    0.187485             0.286925     53.038971
12                   root      trace.to_ts-order-service.duration_p95            24                     2       2024-01-23T11:22:00Z     2024-01-23T11:23:00Z    0.202128             0.297560     47.213869
13        ts-seat-service     trace.to_ts-order-service.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:00:00Z    0.010561             0.015085     42.836344
14     ts-travel2-service      trace.to_ts-train-service.duration_p95            25                     2       2024-01-23T11:14:00Z     2024-01-23T11:17:00Z    0.488604             0.676625     38.481251
15    ts-security-service     trace.to_ts-order-service.duration_mean            25                     2       2024-01-23T10:59:00Z     2024-01-23T11:05:00Z    0.053018             0.068592     29.374702
16                   root      trace.to_ts-auth-service.duration_mean            25                     2       2024-01-23T11:00:00Z     2024-01-23T11:02:00Z    0.448144             0.575149     28.340059
17       ts-route-service  trace.from_ts-travel-service.duration_mean            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.007321             0.009320     27.303131
18      ts-travel-service     trace.to_ts-route-service.duration_mean            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.007321             0.009320     27.303131
19       ts-basic-service     trace.to_ts-route-service.duration_mean            25                     2       2024-01-23T11:04:00Z     2024-01-23T11:05:00Z    0.011674             0.014787     26.666916, 136, 196)```
```